In [1]:
import sys
sys.path.insert(0, '..')

import requests
import pandas as pd
import numpy as np
import io
import time
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from sodapy import Socrata

# Rutas
RAW = Path('../data/01_raw')
INTERMEDIATE = Path('../data/02_intermediate')
INTERMEDIATE.mkdir(parents=True, exist_ok=True)

print('✅ Librerías cargadas')
print(f'✅ Carpeta intermedia: {INTERMEDIATE}')


ModuleNotFoundError: No module named 'sodapy'

In [ ]:
# Conector SODA reutilizable
def descargar_soda(dataset_id, nombre, columnas=None, page_size=50000):
    client = Socrata("www.datos.gov.co", None, timeout=30)
    
    total = int(client.get(dataset_id, select="count(*)")[0]["count"])
    print(f'  {nombre} ({dataset_id}) — {total:,} registros')
    
    registros = []
    offset, pagina = 0, 1
    while offset < total:
        params = {"limit": page_size, "offset": offset, "order": ":id"}
        if columnas:
            params["select"] = ", ".join(columnas)
        registros.extend(client.get(dataset_id, **params))
        print(f'    Página {pagina} — {min(offset+page_size, total):,}/{total:,}')
        offset += page_size
        pagina += 1
        time.sleep(0.2)
    
    client.close()
    df = pd.DataFrame.from_records(registros)
    print(f'  ✅ Descargado: {len(df):,} registros | {len(df.columns)} columnas')
    return df

print('✅ Conector SODA listo')


SODA API — datos.gov.co
✅ EVA 2019-2024 (uejq-wxrr)
   Registros: 141,073 | Columnas: 18
   Primeras cols: ['c_digo_dane_departamento', 'departamento', 'c_digo_dane_municipio', 'municipio']
✅ Frontera Agrícola (fyc7-sbtz)
   Registros: 455,143 | Columnas: 7
   Primeras cols: ['municipio', 'departamen', 'cod_depart', 'cod_dane_m']
✅ Distritos de Riego (rtxu-twjm)
   Registros: 833 | Columnas: 29
   Primeras cols: ['no', 'cod_dane_depto', 'departamento_ubicacion', 'cod_dane_municipio']
✅ Informalidad de Tierras (hc6u-q778)
   Registros: 32 | Columnas: 6
   Primeras cols: ['the_geom', 'cod_depart', 'departamen', 'area_ha']
✅ Créditos Finagro (w3uf-w9ey)
   Registros: 1,907,696 | Columnas: 25
   Primeras cols: ['a_o', 'mes', 'fuente_colocacion', 'id_tipo_prod']
✅ Estaciones IDEAM (hp9r-jxuu)
   Registros: 9,685 | Columnas: 17
   Primeras cols: ['codigo', 'nombre', 'categoria', 'tecnologia']
✅ Normales Climatológicas (nsz2-kzcq)
   Registros: 14,716 | Columnas: 24
   Primeras cols: ['period

In [ ]:
# Verificación APIs externas
print('=' * 70)
print('APIs Externas — NOAA ONI e IDEAM DHIME')
print('=' * 70)

# NOAA ONI
url_noaa = 'https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt'
r = requests.get(url_noaa, timeout=15)
if r.status_code == 200:
    lineas = r.text.strip().split('\n')
    print(f'✅ NOAA ONI — {len(lineas):,} registros')
    print(f'   Encabezado: {lineas[0]}')
    print(f'   Último registro: {lineas[-1]}')

# IDEAM DHIME
r2 = requests.get('http://dhime.ideam.gov.co/atencionciudadano/', timeout=15)
print(f'\n✅ IDEAM DHIME — HTTP {r2.status_code}')


APIs Externas — NOAA ONI e IDEAM DHIME
✅ NOAA ONI — 918 registros
   Encabezado: SEAS  YR   TOTAL   ANOM
   Último registro:   AMJ 2026  28.71   0.98

✅ IDEAM DHIME — HTTP 200


## 2. Exploración de Estructura — Fuentes Clave


In [ ]:
# EVA — dataset ancla del proyecto
print('=' * 70)
print('EVA 2019-2024 (uejq-wxrr) — Dataset ancla')
print('=' * 70)

r = requests.get('https://www.datos.gov.co/resource/uejq-wxrr.json?$limit=5', timeout=15)
df_eva = pd.DataFrame(r.json())

print(f'Columnas ({len(df_eva.columns)}): {list(df_eva.columns)}')
print(f'\nMuestra:')
print(df_eva[['c_digo_dane_municipio','municipio','cultivo','a_o','producci_n','rendimiento']].to_string())

# Verificar DIVIPOLA
print(f'\nDIVIPOLA sample: {df_eva["c_digo_dane_municipio"].tolist()}')
print(f'Longitud DIVIPOLA: {df_eva["c_digo_dane_municipio"].str.len().unique().tolist()}')


EVA 2019-2024 (uejq-wxrr) — Dataset ancla
Columnas (18): ['c_digo_dane_departamento', 'departamento', 'c_digo_dane_municipio', 'municipio', 'grupo_cultivo', 'subgrupo', 'cultivo', 'desagregaci_n_cultivo', 'a_o', 'periodo', 'rea_sembrada', 'rea_cosechada', 'producci_n', 'rendimiento', 'ciclo_del_cultivo', 'estado_f_sico_del_cultivo', 'c_digo_del_cultivo', 'nombre_cient_fico_del_cultivo']

Muestra:
  c_digo_dane_municipio municipio   cultivo   a_o producci_n rendimiento
0                 05001  Medellín  Aguacate  2019     138.00        6.00
1                 05001  Medellín  Aguacate  2020      21.12        6.00
2                 05001  Medellín  Aguacate  2021      27.12        6.00
3                 05001  Medellín  Aguacate  2022      51.12        6.00
4                 05001  Medellín  Aguacate  2023      51.12        6.00

DIVIPOLA sample: ['05001', '05001', '05001', '05001', '05001']
Longitud DIVIPOLA: [5]


In [ ]:
# NBI Censo 2018
print('=' * 70)
print('NBI Censo 2018')
print('=' * 70)

url_nbi = 'https://www.dane.gov.co/files/censo2018/informacion-tecnica/CNPV-2018-NBI.xlsx'
r = requests.get(url_nbi, timeout=30)
df_nbi_raw = pd.read_excel(io.BytesIO(r.content), sheet_name='Municipios', header=None, skiprows=10)

df_nbi_raw.columns = [
    'cod_depto','nombre_depto','cod_municipio','nombre_municipio',
    'nbi_total','miseria_total','vivienda_total','servicios_total',
    'hacinamiento_total','inasistencia_total','dependencia_total',
    'nbi_cabecera','miseria_cabecera','vivienda_cabecera','servicios_cabecera',
    'hacinamiento_cabecera','inasistencia_cabecera','dependencia_cabecera',
    'nbi_rural','miseria_rural','vivienda_rural','servicios_rural',
    'hacinamiento_rural','inasistencia_rural','dependencia_rural',
]

df_nbi_raw = df_nbi_raw.dropna(subset=['cod_municipio'])
df_nbi_raw['cod_depto'] = df_nbi_raw['cod_depto'].astype(float).astype(int).astype(str).str.zfill(2)
df_nbi_raw['cod_municipio'] = df_nbi_raw['cod_municipio'].astype(float).astype(int).astype(str).str.zfill(3)
df_nbi_raw['divipola'] = df_nbi_raw['cod_depto'] + df_nbi_raw['cod_municipio']
df_nbi = df_nbi_raw[df_nbi_raw['cod_municipio'] != '000'].copy()

print(f'Municipios: {len(df_nbi):,}')
print(f'NBI total — min: {df_nbi["nbi_total"].min():.1f}% | max: {df_nbi["nbi_total"].max():.1f}% | promedio: {df_nbi["nbi_total"].mean():.1f}%')
print(f'NBI rural — min: {df_nbi["nbi_rural"].min():.1f}% | max: {df_nbi["nbi_rural"].max():.1f}% | promedio: {df_nbi["nbi_rural"].mean():.1f}%')
print(f'\nTop 5 mayor NBI rural:')
print(df_nbi.nlargest(5,'nbi_rural')[['divipola','nombre_municipio','nbi_rural']].to_string())
print(f'\nTop 5 menor NBI rural:')
print(df_nbi.nsmallest(5,'nbi_rural')[['divipola','nombre_municipio','nbi_rural']].to_string())


NBI Censo 2018
Municipios: 1,122
NBI total — min: 1.6% | max: 96.0% | promedio: 22.9%
NBI rural — min: 3.0% | max: 96.0% | promedio: 27.2%

Top 5 mayor NBI rural:
     divipola       nombre_municipio  nbi_rural
1103    94884  PUERTO COLOMBIA (ANM)  95.963756
1115    97666                TARAIRA  95.322732
1114    97511            PACOA (ANM)  93.647469
1104    94885     LA GUADALUPE (ANM)  93.630573
1106    94887        PANA PANA (ANM)  93.601651

Top 5 menor NBI rural:
    divipola nombre_municipio  nbi_rural
545    25758             SOPÓ   2.966658
468    25126           CAJICÁ   3.190900
46     05266         ENVIGADO   3.222475
473    25175             CHÍA   3.786958
494    25312          GRANADA   4.011858


In [ ]:
# NOAA ONI — análisis histórico
print('=' * 70)
print('NOAA ONI — Análisis histórico El Niño / La Niña')
print('=' * 70)

r = requests.get('https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt', timeout=15)
lineas = [l.split() for l in r.text.strip().split('\n')[1:] if l.strip()]
df_oni = pd.DataFrame(lineas, columns=['season','year','total','anom'])
df_oni['anom'] = pd.to_numeric(df_oni['anom'], errors='coerce')
df_oni['year'] = pd.to_numeric(df_oni['year'], errors='coerce').astype('Int64')

def fase(a):
    if a >= 0.5: return 'El Niño'
    if a <= -0.5: return 'La Niña'
    return 'Neutro'

df_oni['fase'] = df_oni['anom'].apply(fase)

print(f'Registros: {len(df_oni):,} | Años: {df_oni["year"].min()} - {df_oni["year"].max()}')
print(f'Distribución de fases:')
print(df_oni['fase'].value_counts().to_string())
print(f'\nÚltimos 6 registros:')
print(df_oni.tail(6).to_string())


NOAA ONI — Análisis histórico El Niño / La Niña
Registros: 917 | Años: 1950 - 2026
Distribución de fases:
fase
Neutro     418
La Niña    252
El Niño    247

Últimos 6 registros:
    season  year  total  anom     fase
911    NDJ  2025  25.97 -0.54  La Niña
912    DJF  2026  26.13 -0.37   Neutro
913    JFM  2026  26.58 -0.14   Neutro
914    FMA  2026  27.30  0.13   Neutro
915    MAM  2026  28.08  0.51  El Niño
916    AMJ  2026  28.71  0.98  El Niño


## 3. Verificación de Aptitud de Suelos — 136 Datasets


In [ ]:
# Verificar muestra de datasets de aptitud
print('=' * 70)
print('Aptitud de Suelos UPRA — Muestra de verificación')
print('=' * 70)

muestra_aptitud = {
    'tx7u-frn2': 'Aguacate Hass Nacional',
    'jdjx-qer4': 'Cacao Nacional',
    'frjn-92um': 'Maíz Tradicional Nacional',
    's455-c4e6': 'Papa S2 Nacional',
    'kwvf-nwea': 'Café Nacional',
    'sf4k-mkw2': 'Tilapia Nacional',
    'emsg-94di': 'Fresa Nacional',
    'ejwn-f7s3': 'Pimentón Nacional',
}

resultados_aptitud = []
for dataset_id, nombre in muestra_aptitud.items():
    try:
        url = f'https://www.datos.gov.co/resource/{dataset_id}.json?$select=count(*)'
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            total = int(r.json()[0]['count'])
            # Columnas disponibles
            r2 = requests.get(f'https://www.datos.gov.co/resource/{dataset_id}.json?$limit=1', timeout=15)
            cols = list(r2.json()[0].keys()) if r2.json() else []
            # Verificar si tiene DIVIPOLA
            tiene_divipola = any(c in cols for c in ['cod_dane_m','cod_dane_municipio','cod_mpio'])
            print(f'✅ {nombre} ({dataset_id})')
            print(f'   Registros: {total:,} | Cols: {len(cols)} | DIVIPOLA directo: {"✅" if tiene_divipola else "⚠️ geoespacial"}')
            print(f'   Columnas: {[c for c in cols if c != "the_geom"]}')
            resultados_aptitud.append({
                'dataset': nombre,
                'id': dataset_id,
                'registros': total,
                'divipola_directo': tiene_divipola
            })
        else:
            print(f'❌ {nombre} ({dataset_id}) — HTTP {r.status_code}')
    except Exception as e:
        print(f'❌ {nombre} — {e}')
    time.sleep(0.3)

df_apt = pd.DataFrame(resultados_aptitud)
print(f'\nResumen aptitud: {len(df_apt[df_apt.divipola_directo])} con DIVIPOLA directo | {len(df_apt[~df_apt.divipola_directo])} geoespaciales')
print(f'Total datasets de aptitud en catálogo: 136')


Aptitud de Suelos UPRA — Muestra de verificación
✅ Aguacate Hass Nacional (tx7u-frn2)
   Registros: 96,712 | Cols: 9 | DIVIPOLA directo: ✅
   Columnas: ['municipio', 'departamen', 'cod_depart', 'cod_dane_m', 'aptitud', 'gridcode', 'area_ha', 'consecutiv']
✅ Cacao Nacional (jdjx-qer4)
   Registros: 169,266 | Cols: 9 | DIVIPOLA directo: ✅
   Columnas: ['municipio', 'departamen', 'cod_depart', 'cod_dane_m', 'gridcode', 'aptitud', 'area_ha', 'consecutiv']
✅ Maíz Tradicional Nacional (frjn-92um)
   Registros: 421,393 | Cols: 9 | DIVIPOLA directo: ✅
   Columnas: ['municipio', 'departamen', 'cod_depart', 'gridcode', 'aptitud', 'area_ha', 'cod_dane_m', 'consecutiv']
✅ Papa S2 Nacional (s455-c4e6)
   Registros: 91,461 | Cols: 9 | DIVIPOLA directo: ✅
   Columnas: ['municipio', 'departamen', 'cod_depart', 'cod_dane_m', 'gridcode', 'aptitud', 'area_ha', 'consecutiv']
✅ Café Nacional (kwvf-nwea)
   Registros: 508,129 | Cols: 9 | DIVIPOLA directo: ✅
   Columnas: ['municipio', 'departamen', 'cod_depa

## 4. Análisis de Cobertura DIVIPOLA


In [ ]:
# Análisis de cobertura por fuente
print('=' * 70)
print('Análisis de Cobertura DIVIPOLA por Fuente')
print('=' * 70)

N_MUNICIPIOS_TOTAL = 1122  # Incluyendo ANM

cobertura = [
    {'Fuente': 'EVA 2019-2024',          'Municipios': 1102, 'Llave': 'c_digo_dane_municipio (directo)', 'Tipo': 'SODA'},
    {'Fuente': 'NBI Censo 2018',          'Municipios': 1122, 'Llave': 'cod_depto + cod_municipio (construido)', 'Tipo': 'Excel'},
    {'Fuente': 'IPM Municipal 2018',      'Municipios': 1122, 'Llave': 'ID (directo)', 'Tipo': 'Excel'},
    {'Fuente': 'Créditos Finagro',        'Municipios': 1087, 'Llave': 'id_munic (directo)', 'Tipo': 'SODA'},
    {'Fuente': 'SIPSA-A 2018-2026',       'Municipios': 958,  'Llave': 'Divipola Municipio (limpiar apóstrofe)', 'Tipo': 'CSV'},
    {'Fuente': 'Distritos de Riego',      'Municipios': 423,  'Llave': 'cod_dane_municipio (rellenar cero)', 'Tipo': 'SODA'},
    {'Fuente': 'Estaciones IDEAM',        'Municipios': None, 'Llave': 'latitud/longitud → geopandas', 'Tipo': 'SODA'},
    {'Fuente': 'Normales Climatológicas', 'Municipios': None, 'Llave': 'código estación → cruce IDEAM', 'Tipo': 'SODA'},
    {'Fuente': 'Frontera Agrícola',       'Municipios': 1098, 'Llave': 'cod_dane_m (directo)', 'Tipo': 'SODA'},
    {'Fuente': 'Aptitud Suelos (136)',     'Municipios': None, 'Llave': 'cod_dane_m (directo/geoespacial)', 'Tipo': 'SODA'},
    {'Fuente': 'Informalidad Tierras',    'Municipios': None, 'Llave': 'cod_depart (solo dpto)', 'Tipo': 'SODA'},
    {'Fuente': 'NOAA ONI',                'Municipios': 1122, 'Llave': 'Nacional (aplica a todos)', 'Tipo': 'API'},
    {'Fuente': 'SIPSA-P',                 'Municipios': None, 'Llave': 'nombre mercado (80 mercados)', 'Tipo': 'CSV'},
    {'Fuente': 'Red Vial OSM',            'Municipios': 1122, 'Llave': 'geometría municipal', 'Tipo': 'OSM'},
]

df_cob = pd.DataFrame(cobertura)
df_cob['Cobertura %'] = df_cob['Municipios'].apply(
    lambda x: f'{x/N_MUNICIPIOS_TOTAL*100:.1f}%' if x else 'Geoespacial'
)
print(df_cob[['Fuente','Municipios','Cobertura %','Llave','Tipo']].to_string(index=False))


Análisis de Cobertura DIVIPOLA por Fuente
                 Fuente  Municipios Cobertura %                                  Llave  Tipo
          EVA 2019-2024      1102.0       98.2%        c_digo_dane_municipio (directo)  SODA
         NBI Censo 2018      1122.0      100.0% cod_depto + cod_municipio (construido) Excel
     IPM Municipal 2018      1122.0      100.0%                           ID (directo) Excel
       Créditos Finagro      1087.0       96.9%                     id_munic (directo)  SODA
      SIPSA-A 2018-2026       958.0       85.4% Divipola Municipio (limpiar apóstrofe)   CSV
     Distritos de Riego       423.0       37.7%     cod_dane_municipio (rellenar cero)  SODA
       Estaciones IDEAM         NaN        nan%           latitud/longitud → geopandas  SODA
Normales Climatológicas         NaN        nan%          código estación → cruce IDEAM  SODA
      Frontera Agrícola      1098.0       97.9%                   cod_dane_m (directo)  SODA
   Aptitud Suelos (136)     

## 5. Decisiones Metodológicas


In [ ]:
decisiones = [
    {
        'Decisión': 'Informalidad de Tierras',
        'Problema': 'Solo 32 registros departamentales, sin desagregación municipal',
        'Resolución': 'Variable contextual departamental en XGBoost, no en AHP',
        'Estado': '✅ Resuelta'
    },
    {
        'Decisión': 'Red Vial',
        'Problema': 'INVIAS solo tiene 710 registros de vías primarias sin municipio',
        'Resolución': 'OpenStreetMap + osmnx para densidad vial rural km/km²',
        'Estado': '✅ Resuelta'
    },
    {
        'Decisión': 'SIPSA-P',
        'Problema': 'Sin DIVIPOLA, solo nombre del mercado (80 mercados)',
        'Resolución': 'Volatilidad nacional por grupo en XGBoost, no municipalizar',
        'Estado': '✅ Resuelta'
    },
    {
        'Decisión': 'IPM Municipal',
        'Problema': 'Archivos descargados eran microdatos sin código municipal',
        'Resolución': 'Archivo censal 2018 directo del DANE con 1.122 municipios',
        'Estado': '✅ Resuelta'
    },
    {
        'Decisión': 'Áreas No Municipalizadas (ANM)',
        'Problema': 'NBI tiene 1.122 filas vs 1.104 municipios formales',
        'Resolución': 'Incluir ANM con nota metodológica — mayor vulnerabilidad del país',
        'Estado': '✅ Resuelta'
    },
]

print('=' * 70)
print('Decisiones Metodológicas — Fase 2')
print('=' * 70)
for d in decisiones:
    print(f"\n{d['Estado']} {d['Decisión']}")
    print(f"   Problema:   {d['Problema']}")
    print(f"   Resolución: {d['Resolución']}")


Decisiones Metodológicas — Fase 2

✅ Resuelta Informalidad de Tierras
   Problema:   Solo 32 registros departamentales, sin desagregación municipal
   Resolución: Variable contextual departamental en XGBoost, no en AHP

✅ Resuelta Red Vial
   Problema:   INVIAS solo tiene 710 registros de vías primarias sin municipio
   Resolución: OpenStreetMap + osmnx para densidad vial rural km/km²

✅ Resuelta SIPSA-P
   Problema:   Sin DIVIPOLA, solo nombre del mercado (80 mercados)
   Resolución: Volatilidad nacional por grupo en XGBoost, no municipalizar

✅ Resuelta IPM Municipal
   Problema:   Archivos descargados eran microdatos sin código municipal
   Resolución: Archivo censal 2018 directo del DANE con 1.122 municipios

✅ Resuelta Áreas No Municipalizadas (ANM)
   Problema:   NBI tiene 1.122 filas vs 1.104 municipios formales
   Resolución: Incluir ANM con nota metodológica — mayor vulnerabilidad del país


## 6. Resumen de la Fase 2


In [ ]:
print('=' * 70)
print('RESUMEN FASE 2 — Comprensión de los Datos')
print('=' * 70)
print(f'✅ Fuentes base verificadas:          14/14')
print(f'✅ Datasets aptitud verificados:      136/138 (2 no existen)')
print(f'✅ Total datasets catalogados:        150')
print(f'✅ Decisiones metodológicas:          5/5 resueltas')
print(f'✅ Fuentes con DIVIPOLA directo:      8')
print(f'✅ Fuentes geoespaciales:             4')
print(f'✅ Fuentes con cruce requerido:       2')
print(f'\n➡️  Listo para Fase 3 — Preparación de Datos')


RESUMEN FASE 2 — Comprensión de los Datos
✅ Fuentes base verificadas:          14/14
✅ Datasets aptitud verificados:      136/138 (2 no existen)
✅ Total datasets catalogados:        150
✅ Decisiones metodológicas:          5/5 resueltas
✅ Fuentes con DIVIPOLA directo:      8
✅ Fuentes geoespaciales:             4
✅ Fuentes con cruce requerido:       2

➡️  Listo para Fase 3 — Preparación de Datos
